In [3]:
import pandas as pd
import yaml

In [5]:
mapping = yaml.safe_load(open("mappings/mapping.yaml", "r"))
mapping

{'Biomass CHP': {'dataset': [{'name': 'heat and power co-generation, wood chips, 2000 kW',
    'reference product': 'electricity, high voltage',
    'unit': 'kilowatt hour'},
   {'name': 'heat and power co-generation, wood chips, 2000 kW, state-of-the-art 2014',
    'reference product': 'electricity, high voltage',
    'unit': 'kilowatt hour'},
   {'name': 'heat and power co-generation, wood chips, 6667 kW',
    'reference product': 'electricity, high voltage',
    'unit': 'kilowatt hour'},
   {'name': 'heat and power co-generation, wood chips, 6667 kW, state-of-the-art 2014',
    'reference product': 'electricity, high voltage',
    'unit': 'kilowatt hour'}],
  'scenario variable': 'SE|Electricity|Biomass|++|Combined Heat and Power w/o CC'},
 'Biomass IGCC': {'dataset': [{'name': 'electricity production, at biomass-fired IGCC power plant',
    'reference product': 'electricity, high voltage',
    'unit': 'kilowatt hour'}],
  'scenario variable': 'SE|Electricity|Biomass|++|Gasification

In [16]:
names = []
refprods = []
units = []
scenvar = []
premise_name = []
for k, ddict in mapping.items():
    for ds in ddict["dataset"]:
        names.append(ds["name"])
        refprods.append(ds["reference product"])
        units.append(ds["unit"])
        scenvar.append(ddict["scenario variable"])
        premise_name.append(k)

df = pd.DataFrame(
    {
        "dataset name": names,
        "dataset reference product": refprods,
        "dataset unit": units,
        "scenario variable": scenvar,
        "premise name": premise_name
    }
)

In [26]:
sel = df[df["scenario variable"].str.startswith("FE|")]

In [27]:
def fill_REMIND_index(x):
    sector = get_sector(x)
    entyfe = get_entyfe(x, sector)

def get_sector(x):
    if "|Transport|" in x:
        return "trans"
    elif "|Industry|" in x:
        return "indst"
    elif "|Buildings|" in x:
        return "build"
    elif "|CDR|" in x:
        return "cdr"
    else:
        return ""
    
def get_entyfe(x):
    cats = x.split(" - ")
    if len(cats) <= 1:
        return ""
    else:
        return cats[-1]


In [28]:
sel["sector"] = sel["scenario variable"].apply(get_sector)
sel["fuel"] = sel["premise name"].apply(get_entyfe)
sel["REMIND index"] = sel["sector"] + " - " + sel["fuel"]
sel["share"] = "regional"

/p/tmp/davidba/anaconda/ipykernel_3290704/3637779058.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel["sector"] = sel["scenario variable"].apply(get_sector)
/p/tmp/davidba/anaconda/ipykernel_3290704/3637779058.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel["fuel"] = sel["premise name"].apply(get_entyfe)
/p/tmp/davidba/anaconda/ipykernel_3290704/3637779058.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] 

In [29]:
sel.sort_values("scenario variable")[
    ["REMIND index", "scenario variable", "dataset name", "dataset reference product", "dataset unit", "share"]
    ].to_csv("mappings/demFE_v0.csv", sep=";", index=False)